# Module 19 Lab — AutoGen / AG2: Iterative Code Review with Nested Chats

**Goal:** Build a multi-agent code review pipeline using AutoGen v0.2's nested chat feature.

The pipeline works in three layers:
1. **Developer** writes code from a specification
2. **Reviewer** sub-conversation fires automatically (nested chat) and checks code quality
3. **Security Auditor** sub-conversation fires automatically after the reviewer and checks for vulnerabilities
4. **Orchestrator** (`UserProxyAgent`) manages the outer conversation and terminates when it sees `APPROVED: Developer`

By the end of this lab you will be able to:
- Define agents with targeted system messages
- Register nested chats with `register_nested_chats()`
- Write trigger functions that extract context from the message history
- Inspect and parse the full conversation trace

In [ ]:
!pip install pyautogen

In [ ]:
import os
import json
import datetime

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")

# Use whichever key you have
if OPENAI_API_KEY:
    llm_config = {
        "config_list": [{"model": "gpt-4o-mini", "api_key": OPENAI_API_KEY}],
        "temperature": 0,
    }
elif ANTHROPIC_API_KEY:
    llm_config = {
        "config_list": [{
            "model": "claude-haiku-4-5-20251001",
            "api_key": ANTHROPIC_API_KEY,
            "api_type": "anthropic",
        }],
        "temperature": 0,
    }
else:
    raise ValueError("Set OPENAI_API_KEY or ANTHROPIC_API_KEY in environment")

print("LLM config ready:", llm_config["config_list"][0]["model"])

## Section 1 — System Messages

System messages are the primary lever for shaping agent behaviour in AutoGen. A few rules:

- **Be specific about what to produce.** Vague instructions produce vague outputs.
- **Include the termination string in every agent that might end the conversation.** AutoGen's `is_termination_msg` checks the *content* field of any message; if the developer never writes the trigger phrase, the chat runs until `max_consecutive_auto_reply`.
- **Keep roles orthogonal.** Each agent should own exactly one concern — correctness, style, or security — so nested chats don't duplicate work.

The three messages below are already written. Read them carefully before moving on.

In [ ]:
DEVELOPER_SYSTEM_MSG = """You are an expert Python developer.
When given a specification, write clean, working Python code.
After the reviewer and security auditor approve, write exactly: APPROVED: Developer"""

REVIEWER_SYSTEM_MSG = """You are a senior code reviewer. Review the code for:
- Correctness and logic errors
- PEP 8 compliance
- Proper error handling
- Clear variable naming
Write 'LOOKS GOOD' if the code passes review, or list specific issues."""

SECURITY_MSG = """You are a security auditor. Review code for:
- SQL injection risks
- Command injection (use of eval, exec, os.system)
- Hardcoded secrets or credentials
- Unsafe deserialization
Write 'SECURITY APPROVED' if no issues found, or list specific vulnerabilities."""

print("System messages defined.")

## Section 2 — Agent Definitions

AutoGen v0.2 has two main agent types:

| Type | Purpose |
|---|---|
| `AssistantAgent` | LLM-backed; responds to messages using its system message |
| `UserProxyAgent` | Represents the human or orchestrator; can run code, call tools, or just relay messages |

**TODO 1–4:** Create the four agents below. The import and skeleton are provided.

Key parameters:
- `human_input_mode="NEVER"` — the orchestrator runs fully autonomously
- `max_consecutive_auto_reply=6` — safety ceiling so the chat doesn't run forever
- `is_termination_msg` — a callable that returns `True` when the conversation should stop
- `code_execution_config=False` — we are *reviewing* code, not *running* it

In [ ]:
from autogen import AssistantAgent, UserProxyAgent

# TODO 1: Create the developer agent
# developer = AssistantAgent(
#     name="Developer",
#     system_message=DEVELOPER_SYSTEM_MSG,
#     llm_config=llm_config,
# )
developer = None  # replace with AssistantAgent(...)

# TODO 2: Create the reviewer agent
# reviewer = AssistantAgent(
#     name="Reviewer",
#     system_message=REVIEWER_SYSTEM_MSG,
#     llm_config=llm_config,
# )
reviewer = None  # replace with AssistantAgent(...)

# TODO 3: Create the security auditor agent
# security_auditor = AssistantAgent(
#     name="SecurityAuditor",
#     system_message=SECURITY_MSG,
#     llm_config=llm_config,
# )
security_auditor = None  # replace with AssistantAgent(...)

# TODO 4: Create the orchestrator (UserProxyAgent)
# orchestrator = UserProxyAgent(
#     name="Orchestrator",
#     human_input_mode="NEVER",
#     max_consecutive_auto_reply=6,
#     is_termination_msg=lambda msg: "APPROVED: Developer" in msg.get("content", ""),
#     code_execution_config=False,
# )
orchestrator = None  # replace with UserProxyAgent(...)

# === SOLUTION (reveal only if stuck) ===
# developer = AssistantAgent(
#     name="Developer",
#     system_message=DEVELOPER_SYSTEM_MSG,
#     llm_config=llm_config,
# )
# reviewer = AssistantAgent(
#     name="Reviewer",
#     system_message=REVIEWER_SYSTEM_MSG,
#     llm_config=llm_config,
# )
# security_auditor = AssistantAgent(
#     name="SecurityAuditor",
#     system_message=SECURITY_MSG,
#     llm_config=llm_config,
# )
# orchestrator = UserProxyAgent(
#     name="Orchestrator",
#     human_input_mode="NEVER",
#     max_consecutive_auto_reply=6,
#     is_termination_msg=lambda msg: "APPROVED: Developer" in msg.get("content", ""),
#     code_execution_config=False,
# )

print("Agents ready.")

## Section 3 — Nested Chat Registration

Nested chats are the key AutoGen feature used in this lab. When you call `register_nested_chats()` on an agent:

1. Every time the **trigger agent** sends a message in the outer conversation, the nested chat fires.
2. The nested chat runs as a **separate sub-conversation** between the host agent and the `recipient`.
3. When it finishes, AutoGen injects the **summary** (controlled by `summary_method`) back into the outer chat as if the host agent said it.

The `message` parameter accepts a callable: `(recipient, messages, sender, config) -> str`. This gives you access to the full outer message history so you can pull the relevant code snippet.

**Pipeline we are building:**
```
Orchestrator → Developer (outer chat)
                   ↓ triggers
              Orchestrator ↔ Reviewer (nested)
                                  ↓ triggers
                         Orchestrator ↔ SecurityAuditor (nested)
```

**TODO 5:** The reviewer nested chat is partially written. Read it, then complete TODO 6 for the security auditor.

In [ ]:
# TODO 5: Register the reviewer nested chat (given — read carefully before moving on)
def create_review_message(recipient, messages, sender, config):
    """Extract the last message and ask the reviewer to check it."""
    last_msg = messages[-1].get("content", "")
    return f"Please review this code:\n\n{last_msg}"

orchestrator.register_nested_chats(
    chat_queue=[{
        "recipient": reviewer,
        "message": create_review_message,
        "max_turns": 2,
        "summary_method": "last_msg",
    }],
    trigger=developer,
)

# TODO 6: Register the security auditor nested chat
# It should:
#   - fire when the reviewer speaks (trigger=reviewer)
#   - send the last message to the security_auditor for review
#   - run for max 2 turns
#   - use "last_msg" as the summary method

# def create_security_message(recipient, messages, sender, config):
#     last_msg = messages[-1].get("content", "")
#     return f"Please perform a security audit on this code:\n\n{last_msg}"

# orchestrator.register_nested_chats(
#     chat_queue=[{
#         "recipient": security_auditor,
#         "message": create_security_message,
#         "max_turns": 2,
#         "summary_method": "last_msg",
#     }],
#     trigger=reviewer,
# )

# === SOLUTION (reveal only if stuck) ===
# def create_security_message(recipient, messages, sender, config):
#     last_msg = messages[-1].get("content", "")
#     return f"Please perform a security audit on this code:\n\n{last_msg}"
#
# orchestrator.register_nested_chats(
#     chat_queue=[{
#         "recipient": security_auditor,
#         "message": create_security_message,
#         "max_turns": 2,
#         "summary_method": "last_msg",
#     }],
#     trigger=reviewer,
# )

print("Nested chats registered.")

## Section 4 — Run the Pipeline and Save the Trace

The specification below asks the developer to write a database query function — exactly the kind of code a security auditor should scrutinise for SQL injection.

Run the cell and watch the nested chats fire. You should see:
- The outer Developer response
- A `>>>>>>>> USING AUTO REPLY...` block for each nested chat
- The final `APPROVED: Developer` message that terminates the outer conversation

The trace is saved to a timestamped JSON file so you can inspect it in Section 5.

In [ ]:
SPEC = """Write a Python function that:
1. Reads a username from user input
2. Queries a SQLite database for that user's records
3. Returns the results as a list of dicts
Include proper error handling."""

orchestrator.initiate_chat(
    developer,
    message=f"Implement the following specification:\n\n{SPEC}",
)

# Save trace
trace_file = f"review_trace_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(trace_file, "w") as f:
    json.dump(orchestrator.chat_messages.get(developer, []), f, indent=2)
print(f"\nTrace saved to {trace_file}")

## Section 5 — Inspect the Trace

The trace is the raw message list from the outer conversation. Each entry is a dict with at minimum:
- `"role"` — `"user"` or `"assistant"`
- `"content"` — the message text
- `"name"` — which agent sent it (not always present)

Nested chat summaries are injected back into the outer conversation as regular messages, so you can identify them by searching for `LOOKS GOOD` or `SECURITY APPROVED`.

**What to look for:**
1. How many total turns did the outer conversation take?
2. At which turn did the nested chats fire?
3. Did the security auditor catch any SQL injection risk in the first draft?

In [ ]:
# Load and pretty-print the trace
with open(trace_file) as f:
    trace = json.load(f)

print(f"Total messages in outer conversation: {len(trace)}\n")
print("=" * 60)

# Count messages by agent name
from collections import Counter
name_counts = Counter(msg.get("name", msg.get("role", "unknown")) for msg in trace)
print("Messages per agent:")
for name, count in name_counts.most_common():
    print(f"  {name}: {count}")

print("\n" + "=" * 60)
print("Full trace:\n")

for i, msg in enumerate(trace):
    role = msg.get("name", msg.get("role", "unknown"))
    content = msg.get("content", "")
    # Flag nested chat injections
    tag = ""
    if "LOOKS GOOD" in content:
        tag = " [REVIEW NESTED CHAT SUMMARY]"
    elif "SECURITY APPROVED" in content:
        tag = " [SECURITY NESTED CHAT SUMMARY]"
    elif "APPROVED: Developer" in content:
        tag = " [TERMINATION MESSAGE]"
    print(f"[{i}] {role}{tag}")
    print(content[:400] + ("..." if len(content) > 400 else ""))
    print("-" * 40)

## Reflection Questions

Think through these before moving to Module 20:

1. **Termination logic:** What happens if the developer writes `APPROVED: Developer` before the reviewer approves? How would you prevent premature termination?

2. **Nested chat depth:** Could you nest a third level — e.g. a performance profiler that fires after the security auditor? What would the trigger chain look like?

3. **Summary injection:** The `summary_method="last_msg"` injects the reviewer's last message verbatim back into the outer chat. What could go wrong with this? When would `summary_method="reflection_with_llm"` be better?

4. **The SQL injection spec:** Did the security auditor catch the injection risk in the developer's first draft? If not, how would you strengthen the security system message?

---

**Next:** Module 20 — CrewAI Content Production Pipeline with Hierarchical Crews and Flows